<a href="https://colab.research.google.com/github/kinemax-core/Genomic-data-science-R/blob/main/parsing_viral_genome.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
def naive(p, t):
    occurrences = []
    for i in range(len(t) - len(p) + 1):  # loop over alignments
        match = True
        for j in range(len(p)):  # loop over characters
            if t[i+j] != p[j]:  # compare characters
                match = False
                break
        if match:
            occurrences.append(i)  # all chars matched; record
    return occurrences


In [ ]:
def reverseComplement(s):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A', 'N': 'N'}
    t = ''
    for base in s:
        t = complement[base] + t
    return t


In [ ]:
def readGenome(filename):
    genome = ''
    with open(filename, 'r') as f:
        for line in f:
            # ignore header line with genome information
            if not line[0] == '>':
                genome += line.rstrip()
    return genome


In [ ]:
def readFastq(filename):
    sequences = []
    qualities = []
    with open(filename) as fh:
        while True:
            fh.readline()  # skip name line
            seq = fh.readline().rstrip()  # read base sequence
            fh.readline()  # skip placeholder line
            qual = fh.readline().rstrip() # base quality line
            if len(seq) == 0:
                break
            sequences.append(seq)
            qualities.append(qual)
    return sequences, qualities


In [ ]:
def reverse_complement(s):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A', 'N': 'N'}
    return ''.join([complement[base] for base in s])[::-1]

def naive_with_rc(p, t):
    occurrences = []

    # 1. Generate the reverse complement string once
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A', 'N': 'N'}
    p_rc = ''.join([complement[base] for base in p])[::-1]
    p_len = len(p)

    # 2. Slide the window across the text
    for i in range(len(t) - p_len + 1):
        # Slice out the exact current window
        window = t[i : i + p_len]

        # If either matches, 'i' is the starting position on the forward strand
        if window == p or window == p_rc:
            occurrences.append(i)

    return occurrences

In [ ]:
import urllib.request

def download_and_parse_lambda_fixed(url):
    with urllib.request.urlopen(url) as response:
        # Decode bytes to text string safely
        raw_text = response.read().decode('utf-8')

    lines = raw_text.splitlines()
    genome_fragments = []

    for line in lines:
        line = line.strip()
        # Ensure we skip headers and keep everything upper-case
        if not line.startswith('>'):
            genome_fragments.append(line.upper())

    # Join into one single continuous string with NO newlines
    genome = ''.join(genome_fragments)
    return genome

url = "https://d28rh4a8wq0iu5.cloudfront.net/ads1/data/lambda_virus.fa"
lambda_genome = download_and_parse_lambda_fixed(url)

# Re-run your matching function
matches = naive_with_rc("ACTAAGT", lambda_genome)
print("Total occurrences:", len(matches))
print("Leftmost offset:", matches[0])

In [ ]:
# Query pattern from the question
pattern = "ACTAAGT"

# Run our strand-aware naive matching function
matches = naive_with_rc(pattern, lambda_genome)

# Find the total number of occurrences
print("Total occurrences:", len(matches))
print(matches[0])


In [ ]:
lambda_genome = download_and_parse_lambda_fixed(url)
print("DEBUG GENOME LENGTH:", len(lambda_genome))
# This MUST output exactly: 48502

In [ ]:
import urllib.request

def download_and_parse_lambda_fixed(url):
    with urllib.request.urlopen(url) as response:
        raw_text = response.read().decode('utf-8')
    lines = raw_text.splitlines()
    genome_fragments = []
    for line in lines:
        line = line.strip()
        if not line.startswith('>'):
            genome_fragments.append(line.upper())
    return ''.join(genome_fragments)

# 1. Download
url = "https://d28rh4a8wq0iu5.cloudfront.net/ads1/data/lambda_virus.fa"
lambda_genome = download_and_parse_lambda_fixed(url)

print("Verified Length:", len(lambda_genome)) # Target: 48502

# 2. Search
matches = naive_with_rc("ACTAAGT", lambda_genome)

print("Total occurrences:", len(matches))     # Target: 5
print("Leftmost offset:", matches[0])         # Target: 26210

In [ ]:
def naive_with_rc_final(p, t):
    occurrences = []

    # 1. Generate the reverse complement cleanly
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A', 'N': 'N'}
    p_rc = ''.join([complement[base] for base in p])[::-1]
    p_len = len(p)

    # 2. Slice and check windows directly with absolutely no index modifications
    for i in range(len(t) - p_len + 1):
        window = t[i : i + p_len]

        if window == p or window == p_rc:
            occurrences.append(i)

    return occurrences

# Test the newly named function on your parsed genome
matches_final = naive_with_rc_final("ACTAAGT", lambda_genome)

print("--- FORCED RESET RESULTS ---")
print("Total occurrences:", len(matches_final))
print("Leftmost offset:", matches_final[0])

In [ ]:
pattern = "AGGAGGTT"
matches_final = naive_2mm(pattern, lambda_genome)
print("Total occurrences:", len(matches_final))
print("Leftmost offset:", matches_final[0])
print(len(naive_with_rc_final(pattern, lambda_genome)) )
# This MUST output exactly: 48502

In [ ]:
def naive_2mm(p, t):
    """
    Finds occurrences of pattern P in text T allowing up to 2 mismatches.
    Does not consider the reverse complement.
    """
    occurrences = []
    p_len = len(p)
    t_len = len(t)

    # Loop through all valid starting positions in the text
    for i in range(t_len - p_len + 1):
        mismatches = 0
        match_valid = True

        # Compare character by character in the current window
        for j in range(p_len):
            if t[i + j] != p[j]:
                mismatches += 1

            # Early exit: if mismatches exceed 2, abort checking this window
            if mismatches > 2:
                match_valid = False
                break

        # If we finished the loop with 2 or fewer mismatches, record the offset
        if match_valid:
            occurrences.append(i)

    return occurrences


# --- VERIFICATION TEST ---
# Using the example from the quiz description:
# 'ACTTTA' in 'ACTTACTTGATAAAGT' should return [0, 4]
test_pattern = 'TTCAAGCC'
# Run the function with the assignment criteria
ans_matches = naive_2mm("TTCAAGCC", lambda_genome)

print("Total occurrences with up to 2 mismatches:", len(ans_matches))
# Output: 20

In [ ]:
def naive_2mm_forward_only(p, t):
    occurrences = []
    p_len = len(p)
    t_len = len(t)

    for i in range(t_len - p_len + 1):
        mismatches = 0
        match_valid = True

        for j in range(p_len):
            # Comparing directly ONLY to the forward pattern p
            if t[i + j] != p[j]:
                mismatches += 1

            if mismatches > 2:
                match_valid = False
                break

        if match_valid:
            occurrences.append(i)

    return occurrences

# Run this one on your genome
ans = naive_2mm_forward_only("GGCGCGGTGGCTCACGCCTGTAAT", lambda_genome)
print("Forward strand only occurrences:", len(ans))


In [ ]:
import urllib.request

def phred33_to_q(qual_char):
    """Converts a Phred+33 ASCII character to its integer quality score."""
    return ord(qual_char) - 33

def analyze_fastq_quality(url):
    print("Downloading FASTQ file...")
    with urllib.request.urlopen(url) as response:
        raw_text = response.read().decode('utf-8')

    lines = raw_text.splitlines()

    # In FASTQ format, every 4 lines represent one read:
    # Line 1: @Header
    # Line 2: Sequence
    # Line 3: +
    # Line 4: Quality Scores
    qual_lines = [lines[i] for i in range(3, len(lines), 4)]

    num_reads = len(qual_lines)
    read_length = len(qual_lines[0])
    print(f"Loaded {num_reads} reads of length {read_length} bases.")

    # Initialize a list to accumulate quality scores for each cycle position
    position_scores = [0] * read_length

    # Sum up the quality scores across all reads for each position
    for qual_string in qual_lines:
        for pos, char in enumerate(qual_string):
            position_scores[pos] += phred33_to_q(char)

    # Calculate the average quality score for each cycle position
    avg_scores = [total_score / num_reads for total_score in position_scores]

    # Print out the averages for the first few cycles to pinpoint the drop
    print("\n--- Average Quality Scores Per Cycle ---")
    for cycle, avg_q in enumerate(avg_scores[:10]):
        print(f"Cycle {cycle}: Average Phred Q-Score = {avg_q:.2f}")

    # Automatically identify the cycle with the lowest average score
    bad_cycle = avg_scores.index(min(avg_scores))
    print(f"\nIdentified Bad Cycle (0-based index): {bad_cycle}")

    return bad_cycle

# URL provided in your assignment text
fastq_url = "https://d28rh4a8wq0iu5.cloudfront.net/ads1/data/ERR037900_1.first1000.fastq"
analyze_fastq_quality(fastq_url)